# 01 — `HFBState`, line by line

This is the foundation for every later notebook. We will separate three layers:

1. **Python:** what syntax such as `@dataclass`, `@classmethod`, `@property`, `@`, `.T`, and `.conj()` does;
2. **linear algebra:** matrix shapes, adjoints, eigendecompositions, determinants, and Pfaffians;
3. **physics:** quasiparticle vacua, the Thouless state, Slater determinants, densities, sectors, and fidelities.

The source convention is

$$\beta = U^\dagger c + V^\dagger c^\dagger,$$

where $c=(c_0,\ldots,c_{m-1})^T$ contains particle annihilation operators and $m$ is the number of one-particle modes.

In [ ]:
from pathlib import Path
import inspect
import sys
import numpy as np

ROOT = Path.cwd()
while not (ROOT / 'src' / 'NSMFermions').exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError('Run this notebook inside the repository')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src' / 'NSMFermions'))

from hfb import HFBState
print('repository:', ROOT)
print('HFBState source file:', inspect.getsourcefile(HFBState))


## 1. Minimal Python vocabulary

| Source syntax | Python meaning | Mathematical meaning here |
|---|---|---|
| `class HFBState:` | Define a new object type | A container representing one Gaussian state |
| `@dataclass` | Generate `__init__`, `__repr__`, and comparisons from declared fields | Give names to $U,V,Z$ rather than storing an anonymous tuple |
| `@classmethod` | Method receives the class as `cls`; called as `HFBState.from_thouless(...)` | An alternative constructor for the same physical object |
| `@property` | Call a method using attribute syntax | Compute $\rho$ as `state.rho`, without storing a second copy |
| `A @ B` | Matrix multiplication | $AB$ |
| `A * B` | Element-by-element multiplication | Not matrix multiplication |
| `A.T` | Transpose, without complex conjugation | $A^T$ |
| `A.conj()` | Complex conjugation | $A^*$ |
| `A.conj().T` | Hermitian adjoint | $A^\dagger$ |
| `np.eye(m)` | $m\times m$ identity array | $I_m$ |
| `len(U)` | Number of rows of a square array | Number of modes $m$ |
| `raise ValueError(...)` | Stop on invalid input | Refuse a mathematically undefined state |

In [ ]:
A = np.array([[1, 2j], [3, 4]], complex)
print('A =\n', A)
print('A.T =\n', A.T)
print('A.conj() =\n', A.conj())
print('A dagger =\n', A.conj().T)
print('A @ A =\n', A @ A)
print('A * A =\n', A * A)


## 2. The dataclass fields

The source begins conceptually as

```python
@dataclass
class HFBState:
    U: np.ndarray
    V: np.ndarray
    Z: np.ndarray = None
```

The annotation `U: np.ndarray` documents the intended type. Python does not automatically enforce it, so constructors validate important conditions explicitly. `Z=None` means that a finite Thouless matrix was not supplied or may not exist.

For $m$ modes, all three matrices have shape $(m,m)$. `U` and `V` define the quasiparticles. `Z` describes the same state in the particle-vacuum Thouless chart when that chart is finite.

In [ ]:
print(inspect.signature(HFBState))
print('Declared dataclass fields:', list(HFBState.__dataclass_fields__))


## 3. `from_thouless`: construct a paired Gaussian vacuum

The mathematical state is

$$|\Phi(Z)\rangle=\mathcal N(Z)\exp\left(\frac12\sum_{ij}Z_{ij}c_i^\dagger c_j^\dagger\right)|0\rangle.$$

Fermionic creation operators anticommute, so only the antisymmetric part contributes:

$$Z^T=-Z.$$

This state has even number parity: its exponential contains $0,2,4,\ldots$ creation operators.

In [ ]:
print(inspect.getsource(HFBState.from_thouless.__func__))


### Line-by-line meaning

```python
z = np.asarray(z, complex)
```
Convert lists or real arrays into one complex NumPy array. Complex storage is required because rotations and general Bogoliubov transformations carry phases.

```python
if z.ndim != 2 or z.shape[0] != z.shape[1] or not np.isfinite(z).all():
```
Require a finite square matrix. `or` stops as soon as one invalid condition is true. `np.isfinite` rejects `NaN` and infinity.

```python
if not np.allclose(z, -z.T, atol=1e-12):
```
Numerically enforce $Z=-Z^T$. `atol` is an absolute roundoff tolerance. Notice `.T`, not `.conj().T`: fermionic pair antisymmetry is a transpose relation.

```python
metric = np.eye(len(z)) + z.T @ z.conj()
```
Build

$$M=I+Z^T Z^*.$$

Because $Z^T=-Z$, this is the convention-compatible positive Hermitian matrix used to normalize the Bogoliubov transformation.

```python
values, vectors = np.linalg.eigh(metric)
```
For Hermitian $M$, compute $M=Q\Lambda Q^\dagger$. `values` contains the real eigenvalues $\lambda_a$ and columns of `vectors` are eigenvectors. `eigh`, rather than general `eig`, uses Hermiticity and is more stable.

```python
u = (vectors / np.sqrt(values)) @ vectors.conj().T
```
Dividing each eigenvector column by $\sqrt{\lambda_a}$ constructs

$$U=M^{-1/2}=Q\Lambda^{-1/2}Q^\dagger.$$

This avoids explicitly computing a matrix inverse.

```python
return cls(u, z.conj() @ u, Z=z.copy())
```
Set

$$V=Z^*U,$$

construct the `HFBState`, and retain an independent copy of $Z$. `cls(...)` means construct whichever class invoked this classmethod.

In [ ]:
Z = np.zeros((4, 4), complex)
Z[0, 1] = 0.4 + 0.2j
Z[2, 3] = -0.7j
Z -= Z.T
state = HFBState.from_thouless(Z)
print('Z antisymmetry error:', np.linalg.norm(Z + Z.T))
print('U shape:', state.U.shape, 'V shape:', state.V.shape)
print('canonical error:', state.canonical_error())


## 4. Canonical relations: why `canonical_error` matters

The quasiparticles must obey fermionic anticommutation relations. With the repository convention, this requires

$$U^\dagger U+V^\dagger V=I,$$

$$U^T V+V^T U=0.$$

The first equation gives $\{\beta_i,\beta_j^\dagger\}=\delta_{ij}$; the second gives $\{\beta_i,\beta_j\}=0$. The method calculates the Frobenius norm of both residual matrices and returns the larger one. A correctly constructed state should give a value near floating-point precision, about $10^{-15}$ here.

In [ ]:
print(inspect.getsource(HFBState.canonical_error))
normalization_residual = (
    state.U.conj().T @ state.U
    + state.V.conj().T @ state.V
    - np.eye(len(state.U))
)
anomalous_residual = state.U.T @ state.V + state.V.T @ state.U
print('normalization residual norm:', np.linalg.norm(normalization_residual))
print('anomalous residual norm:', np.linalg.norm(anomalous_residual))


## 5. `rho` and `kappa` properties

The normal and anomalous one-body densities are

$$\rho_{ij}=\langle c_j^\dagger c_i\rangle=(V^*V^T)_{ij},$$

$$\kappa_{ij}=\langle c_j c_i\rangle=(V^*U^T)_{ij}.$$

`rho` is Hermitian and its trace is the average particle number. Its eigenvalues lie between zero and one. `kappa` is antisymmetric and measures pairing. These are computed properties, so the state stores only $U,V,Z$ and cannot develop an inconsistent cached density.

In [ ]:
rho = state.rho
kappa = state.kappa
print('rho =\n', rho)
print('kappa =\n', kappa)
print('rho Hermiticity error:', np.linalg.norm(rho-rho.conj().T))
print('kappa antisymmetry error:', np.linalg.norm(kappa+kappa.T))
print('average particle number Tr(rho):', np.trace(rho).real)
print('natural occupations:', np.linalg.eigvalsh(rho))


## 6. `from_slater`: the number-conserving Gaussian boundary

A Slater determinant with $A$ occupied orbitals is

$$|C\rangle=d_1^\dagger\cdots d_A^\dagger|0\rangle,\qquad d_a^\dagger=\sum_i C_{ia}c_i^\dagger,$$

with $C^\dagger C=I_A$. It is Gaussian, but a nonempty Slater determinant has zero overlap with the particle vacuum and therefore cannot be represented by a finite particle-vacuum Thouless matrix. This is why the code treats it as a separate chart.

In [ ]:
print(inspect.getsource(HFBState.from_slater.__func__))


### Line-by-line meaning

`occupied = np.asarray(orbitals, complex)` standardizes the input. The shape is $(m,A)$: rows are physical modes and columns are occupied orbitals. The condition `occupied.conj().T @ occupied == I` is $C^\dagger C=I_A$.

`empty = null_space(occupied.conj().T)` finds $m-A$ orthonormal vectors perpendicular to every occupied orbital. Thus `[occupied, empty]` is a complete one-body basis.

The arrays `u` and `v` begin as zero matrices. For occupied orbitals, the quasiparticle annihilator is hole-like, $\beta_h=d_h^\dagger$, because creating a particle in an already occupied orbital gives zero. For empty orbitals it is particle-like, $\beta_p=d_p$, because annihilating an empty orbital gives zero. The assignments

```python
v[:, :particles] = occupied.conj()
u[:, particles:] = empty
```

encode precisely those two sets of quasiparticles.

In [ ]:
C = np.array([[1, 0], [0, 0], [0, 1], [0, 0]], complex)
slater = HFBState.from_slater(C)
print('C dagger C =\n', C.conj().T @ C)
print('rho =\n', slater.rho)
print('rho idempotency:', np.linalg.norm(slater.rho @ slater.rho-slater.rho))
print('kappa norm:', np.linalg.norm(slater.kappa))
print('canonical error:', slater.canonical_error())


## 7. `thouless_matrix`: stored value or recovery from $U,V$

If a state was created with `from_thouless`, the code returns the stored copy. Otherwise it solves

$$Z=V^*(U^*)^{-1}.$$

It uses `np.linalg.solve` rather than explicitly forming $(U^*)^{-1}$, because solving a linear system is numerically safer. If $U$ is singular or extremely ill-conditioned, this particle-vacuum chart does not exist. That is normal for a nonempty Slater determinant, not a failure of the physical state.

In [ ]:
print('finite-Z recovery error:', np.linalg.norm(state.thouless_matrix-Z))
try:
    slater.thouless_matrix
except ValueError as error:
    print('Slater chart message:', error)


## 8. Occupation amplitudes and the Pfaffian

An occupation configuration $I=(i_1<\cdots<i_{2n})$ denotes

$$|I\rangle=c_{i_1}^\dagger\cdots c_{i_{2n}}^\dagger|0\rangle.$$

For a finite Thouless state, its unnormalized coefficient is

$$\langle I|\exp(\tfrac12c^\dagger Zc^\dagger)|0\rangle=\operatorname{pf}(Z_{I,I}).$$

The Pfaffian is the antisymmetric analogue of a determinant and satisfies $\operatorname{pf}(A)^2=\det(A)$. Odd configurations have zero amplitude because an even Gaussian contains only even particle numbers.

For a Slater determinant, only configurations with exactly $A$ particles survive, and

$$\langle I|C\rangle=\det(C_I).$$

In [ ]:
for occupied in [(), (0,), (0,1), (2,3), (0,1,2,3)]:
    print(occupied, state.occupation_amplitude(occupied, normalized=True))
print('Slater coefficient on (0,2):',
      slater.occupation_amplitude((0,2), normalized=True))
print('Slater coefficient on (0,1):',
      slater.occupation_amplitude((0,1), normalized=True))


### Normalization factor

The unnormalized Thouless exponential has squared norm

$$\|e^{c^\dagger Zc^\dagger/2}|0\rangle\|^2=\sqrt{\det(I+Z^\dagger Z)}.$$

Therefore the ket coefficient is multiplied by

$$\mathcal N(Z)=\det(I+Z^\dagger Z)^{-1/4}.$$

The code uses `slogdet` and an exponential rather than `det(...)**(-0.25)`, because determinants can overflow or underflow while their logarithms remain representable.

## 9. Fixed-sector state, weight, and fidelity

Let $\mathcal I_S$ be a list of determinants belonging to a selected sector, for example fixed neutron and proton numbers. The normalized intrinsic vacuum is

$$|\Phi\rangle=\sum_I a_I|I\rangle.$$

The sector probability is

$$w_S=\sum_{I\in\mathcal I_S}|a_I|^2=\langle\Phi|P_S|\Phi\rangle.$$

The normalized projected state is

$$|\Phi_S\rangle=\frac{P_S|\Phi\rangle}{\sqrt{w_S}}.$$

For a normalized target $|\Psi_S\rangle$,

$$F_{\rm raw}=|\langle\Psi_S|\Phi\rangle|^2,$$

$$F_{\rm projected}=|\langle\Psi_S|\Phi_S\rangle|^2,$$

and therefore $F_{\rm raw}=w_SF_{\rm projected}$.

In [ ]:
sector = [(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)]
target = np.zeros(len(sector), complex)
target[sector.index((0,1))] = 1
weight = state.fixed_sector_weight(sector)
raw_fidelity = state.fixed_sector_fidelity(target, sector)
projected_fidelity = state.fixed_sector_fidelity(
    target, sector, projected=True
)
print('sector weight:', weight)
print('raw fidelity:', raw_fidelity)
print('projected fidelity:', projected_fidelity)
print('F_raw - weight*F_projected:',
      raw_fidelity-weight*projected_fidelity)


## 10. Object-state diagram

```text
antisymmetric Z                         occupied orbitals C
      |                                        |
      | from_thouless                         | from_slater
      v                                        v
             HFBState(U, V, optional Z)
                    |
          +---------+----------+
          |                    |
          v                    v
     rho, kappa       occupation amplitudes
                              |
                    +---------+----------+
                    |                    |
                    v                    v
              sector weight       sector fidelity
```

Later chapters will act on this same `HFBState` with gauge rotations and spatial Euler rotations.

## 11. Checks you should always perform

For any state used in a calculation:

```python
assert state.canonical_error() < 1e-10
assert np.allclose(state.rho, state.rho.conj().T)
assert np.allclose(state.kappa, -state.kappa.T)
assert np.linalg.eigvalsh(state.rho).min() > -1e-10
assert np.linalg.eigvalsh(state.rho).max() < 1+1e-10
```

If the state is supposed to be a Slater determinant, also check

```python
assert np.linalg.norm(state.kappa) < 1e-8
assert np.linalg.norm(state.rho @ state.rho-state.rho) < 1e-8
```

In [ ]:
assert state.canonical_error() < 1e-10
assert np.allclose(state.rho, state.rho.conj().T)
assert np.allclose(state.kappa, -state.kappa.T)
assert np.linalg.eigvalsh(state.rho).min() > -1e-10
assert np.linalg.eigvalsh(state.rho).max() < 1+1e-10
assert np.linalg.norm(slater.kappa) < 1e-8
assert np.linalg.norm(slater.rho @ slater.rho-slater.rho) < 1e-8
print('All chapter checks passed.')


## What comes next

Chapter 02 starts from $\rho$ and $\kappa$ and explains every contraction in `HFBHamiltonian.energy`, then follows the constrained optimizer that produces `HFBResult`.